# 🔬 Hücreden Koda · Uygulama I
### Kan yaymasında hücreleri bilgisayara saydırmak

**TÜBİTAK-BİDEB 2237 · Uygulamalı Eğitimle Sağlık Araştırmalarında Yapay Zekâ ve Hukuki Çerçevesi**
KTÜ İLAFAR · 24 Ekim 2026 · Dr. Öğr. Üyesi Ramazan Özgür Doğan

---

**Nasıl kullanılır?** Her gri kutu bir *kod hücresi*. Sol üstteki **▶** düğmesine bas (ya da hücreye tıklayıp **Shift + Enter**).
Hücreleri **yukarıdan aşağıya sırayla** çalıştır. Kod yazmana gerek yok; istersen sayıları değiştirip ne olduğunu gör.

> 🩺 **Sağlıkçılar için:** Kodu anlamak zorunda değilsin; her adımın *ne yaptığını* açıklamalarda bulacaksın.
> 💻 **Bilgisayarcılar için:** `#@title` ile gizlenmiş hücreleri açıp kodun tamamını inceleyebilirsin.

**Veri:** BCCD (Blood Cell Count and Detection) — 364 kan yayması görüntüsü, uzman kutuları: RBC (alyuvar), WBC (akyuvar), trombosit. Lisans: MIT · https://github.com/Shenggan/BCCD_Dataset

## 0 · Hazırlık
Kullanacağımız araç kutularını (kütüphaneleri) çağırıyoruz. Hepsi Colab'da hazır geliyor.

In [ ]:
#@title ▶ 0 · Kütüphaneleri yükle { display-mode: "form" }
import os, glob, subprocess, warnings, xml.etree.ElementTree as ET
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from skimage import io, filters, morphology, measure, segmentation, feature
from scipy import ndimage as ndi
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.grid": False})
RENK = {"RBC": "#ef233c", "WBC": "#7c3aed", "Platelets": "#f59e0b"}
print("Hazır ✓")

## 1 · Veriyi indir
Veri seti GitHub'da duruyor; tek komutla bilgisayarımıza (aslında Google'ın bilgisayarına) kopyalıyoruz.

In [ ]:
#@title ▶ 1 · BCCD veri setini indir { display-mode: "form" }
if not os.path.exists("BCCD_Dataset"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Shenggan/BCCD_Dataset.git"], check=True)
KOK = "BCCD_Dataset/BCCD/"
dosyalar = sorted(glob.glob(KOK + "JPEGImages/*.jpg"))
print(f"{len(dosyalar)} görüntü indirildi ✓   örnek: {os.path.basename(dosyalar[0])}")

## 2 · Bir görüntü aslında bir sayı tablosudur
Mikroskop fotoğrafını açalım. Bilgisayar için bu fotoğraf, her noktası (**piksel**) için üç sayı tutan dev bir tablo:
**R**(kırmızı), **G**(yeşil), **B**(mavi) — her biri 0 ile 255 arasında.

In [ ]:
img = io.imread(dosyalar[0])
print("Boyut (yükseklik, genişlik, kanal):", img.shape)
print("Sol üstteki pikselin rengi (R, G, B):", img[0, 0])
print("Tam ortadaki pikselin rengi:", img[240, 320])
plt.imshow(img); plt.title("Kan yayması · " + os.path.basename(dosyalar[0])); plt.axis("off"); plt.show()

In [ ]:
#@title ▶ Yakınlaştır: pikselleri ve sayılarını gör { display-mode: "form" }
y, x = 200, 300            # yakınlaştırılacak yer (değiştirebilirsin)
parca = img[y:y + 12, x:x + 12]
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(img); ax[0].add_patch(Rectangle((x, y), 12, 12, fill=False, ec="yellow", lw=2)); ax[0].set_title("Nereye bakıyoruz?"); ax[0].axis("off")
ax[1].imshow(parca, interpolation="nearest"); ax[1].set_title("12 × 12 piksel · sayı = yeşil kanal")
for i in range(12):
    for j in range(12):
        ax[1].text(j, i, parca[i, j, 1], ha="center", va="center", fontsize=6.5, color="white" if parca[i, j, 1] < 150 else "black")
ax[1].axis("off"); plt.tight_layout(); plt.show()

## 3 · Renk kanalları: hücreler hangi kanalda daha belirgin?
Boyalı yaymada alyuvarlar pembe, akyuvarların çekirdeği mor görünür. Kanalları ayrı ayrı çizelim.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6))
ax[0].imshow(img); ax[0].set_title("Renkli")
for k, (ad, cm) in enumerate([("R · kırmızı", "Reds_r"), ("G · yeşil", "Greens_r"), ("B · mavi", "Blues_r")], start=1):
    ax[k].imshow(img[..., k - 1], cmap="gray"); ax[k].set_title(ad)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
plt.figure(figsize=(7, 2.8))
plt.hist(img[..., 1].ravel(), bins=60, color="#16a34a")
plt.title("Yeşil kanal histogramı: solda hücreler (koyu), sağda arka plan (açık)"); plt.xlabel("parlaklık (0–255)"); plt.show()

## 4 · Uzman etiketleri: "doğru cevap" tablosu
Bir uzman her hücrenin etrafına kutu çizmiş. Bu etiketleri bir tabloya (DataFrame) okuyalım.

In [ ]:
#@title ▶ 4 · Etiket dosyalarını (XML) oku { display-mode: "form" }
satirlar = []
for x in sorted(glob.glob(KOK + "Annotations/*.xml")):
    kok = ET.parse(x).getroot(); ad = kok.find("filename").text
    for o in kok.findall("object"):
        b = o.find("bndbox")
        satirlar.append([ad, o.find("name").text] + [int(float(b.find(k).text)) for k in ["xmin", "ymin", "xmax", "ymax"]])
etiket = pd.DataFrame(satirlar, columns=["dosya", "sinif", "xmin", "ymin", "xmax", "ymax"])
etiket.head()

In [ ]:
print("Toplam etiket:", len(etiket)); print(etiket.sinif.value_counts())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
etiket.sinif.value_counts().plot.bar(ax=ax[0], color=[RENK[c] for c in etiket.sinif.value_counts().index]); ax[0].set_title("Sınıf başına hücre")
etiket[etiket.sinif == "RBC"].groupby("dosya").size().plot.hist(ax=ax[1], bins=20, color="#ef233c"); ax[1].set_title("Görüntü başına alyuvar (RBC) sayısı")
plt.tight_layout(); plt.show()

### Önce veriyi temizle 🧹
Sabahki “Algoritmik Temeller” oturumunun kontrol listesi: **eksik değer, tekrar eden satır, hatalı kayıt.**
Herkesin kullandığı açık bir veri setinde bile sürpriz çıkabilir; bakalım.

In [ ]:
print("Eksik değer      :", int(etiket.isna().sum().sum()))
print("Tekrar eden satır:", int(etiket.duplicated().sum()))
hatali = etiket[(etiket.xmax <= etiket.xmin) | (etiket.ymax <= etiket.ymin)]   # genişliği ya da yüksekliği sıfır / eksi
print("Hatalı kutu (boyutu sıfır ya da ters):", len(hatali))
display(hatali)
etiket = etiket.drop(hatali.index).reset_index(drop=True)
print("Temizlenmiş tablo:", len(etiket), "satır ✓")

🎓 Etiketleme sırasında yanlışlıkla tek bir noktaya tıklanmış: kutunun boyutu sıfır. Model bu satırları
“doğru cevap” sanardı. **Veriye bakmadan modele geçmeyin.**

In [ ]:
def kutulari_ciz(ad, ax=None):
    im = io.imread(KOK + "JPEGImages/" + ad); ax = ax or plt.gca(); ax.imshow(im)
    for _, r in etiket[etiket.dosya == ad].iterrows():
        ax.add_patch(Rectangle((r.xmin, r.ymin), r.xmax - r.xmin, r.ymax - r.ymin, fill=False, ec=RENK[r.sinif], lw=2))
    ax.set_title(ad); ax.axis("off")
ad = os.path.basename(dosyalar[0])
plt.figure(figsize=(7, 5.5)); kutulari_ciz(ad); plt.show()
print(etiket[etiket.dosya == ad].sinif.value_counts().to_dict(), " ← uzmanın saydığı")

## 5 · Bilgisayara saydıralım: eşikleme
Fikir çok basit: **hücreler arka plandan koyu**. Bir eşik değeri seçip "bundan koyu olan her şey hücredir" diyoruz.
Eşiği elle değil, **Otsu yöntemi** histogramdan otomatik buluyor.

In [ ]:
g = img[..., 1] / 255.0                         # yeşil kanal (0–1)
esik = filters.threshold_otsu(g)
maske = g < esik                                # koyu = hücre
maske = morphology.remove_small_objects(maske, 120)   # küçük lekeleri at
maske = ndi.binary_fill_holes(maske)            # alyuvarın açık merkezini doldur
maske = morphology.binary_opening(maske, morphology.disk(4))
etiketli = measure.label(maske)
basit = [p for p in measure.regionprops(etiketli) if p.area >= 900]
print(f"Otsu eşiği: {esik:.2f}  ·  bulunan nesne: {len(basit)}  ·  uzman (RBC + WBC): {etiket[(etiket.dosya == ad) & etiket.sinif.isin(['RBC', 'WBC'])].shape[0]}")
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ax[0].imshow(maske, cmap="gray"); ax[0].set_title("Maske (beyaz = hücre)")
ax[1].imshow(segmentation.mark_boundaries(img, etiketli, color=(1, 1, 0))); ax[1].set_title(f"Basit sayım: {len(basit)} nesne")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

🤔 **Sorun ne?** Birbirine **yapışık** hücreleri tek nesne sayıyoruz. Çözüm: *havza (watershed)* yöntemi.
Her hücrenin "merkezini" bulup komşu hücreleri bir su seti gibi ayırıyoruz.

In [ ]:
def hucre_say(img, min_mesafe=36, min_alan=900):
    g = img[..., 1] / 255.0
    m = g < filters.threshold_otsu(g)
    m = morphology.remove_small_objects(m, 120); m = ndi.binary_fill_holes(m)
    m = morphology.binary_opening(m, morphology.disk(4))
    d = ndi.gaussian_filter(ndi.distance_transform_edt(m), 2)      # merkeze uzaklık haritası
    tepe = feature.peak_local_max(d, min_distance=min_mesafe, labels=measure.label(m), exclude_border=False)
    isaret = np.zeros(d.shape, int); isaret[tuple(tepe.T)] = np.arange(1, len(tepe) + 1)
    lab = segmentation.watershed(-d, isaret, mask=m)
    hucreler = [p for p in measure.regionprops(lab) if p.area >= min_alan]
    return len(hucreler), lab

n, lab = hucre_say(img)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
kutulari_ciz(ad, ax[0]); ax[0].set_title("Uzman")
ax[1].imshow(segmentation.mark_boundaries(img, lab, color=(1, 1, 0))); ax[1].set_title(f"Havza ile sayım: {n} hücre"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 6 · Bütün veri setinde sınayalım
İlk 100 görüntüde bilgisayarın saydığını uzmanın saydığıyla karşılaştıralım.

In [ ]:
sonuc = []
for yol in dosyalar[:100]:
    ad_ = os.path.basename(yol)
    n, _ = hucre_say(io.imread(yol))
    uzman = etiket[(etiket.dosya == ad_) & etiket.sinif.isin(["RBC", "WBC"])].shape[0]
    sonuc.append((ad_, uzman, n))
sonuc = pd.DataFrame(sonuc, columns=["dosya", "uzman", "bilgisayar"])
mae = (sonuc.bilgisayar - sonuc.uzman).abs().mean()
plt.figure(figsize=(5.2, 5))
plt.scatter(sonuc.uzman, sonuc.bilgisayar, alpha=0.6, color="#00426a")
lim = [0, max(sonuc.max(numeric_only=True)) + 3]; plt.plot(lim, lim, "--", color="#ef233c", label="mükemmel uyum")
plt.xlabel("uzmanın saydığı"); plt.ylabel("bilgisayarın saydığı"); plt.legend(); plt.title(f"Ortalama mutlak hata: {mae:.1f} hücre")
plt.show()
sonuc.assign(fark=sonuc.bilgisayar - sonuc.uzman).sort_values("fark").tail(3)

🔎 **En büyük farklara bak:** Bazen bilgisayar *haklı*, uzman o hücreleri etiketlememiş! (BCCD'de kenardaki ve kalabalık bölgedeki
alyuvarların bir kısmı etiketsiz.) Ders: **Yapay zekâ ancak etiketleri kadar iyidir** — veri setini kurarken etiket kalitesine yatırım yap.

In [ ]:
en_farkli = sonuc.assign(fark=sonuc.bilgisayar - sonuc.uzman).sort_values("fark").iloc[-1].dosya
n, lab = hucre_say(io.imread(KOK + "JPEGImages/" + en_farkli))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
kutulari_ciz(en_farkli, ax[0]); ax[0].set_title("Uzman etiketleri")
ax[1].imshow(segmentation.mark_boundaries(io.imread(KOK + "JPEGImages/" + en_farkli), lab, color=(1, 1, 0))); ax[1].set_title(f"Bilgisayar: {n}"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 7 · Akyuvarı (WBC) rengiyle bulmak
Akyuvar çekirdeği **mor**: mavi kanal yüksek, yeşil kanal düşük. Basit bir renk kuralı yazalım.

In [ ]:
def akyuvar_bul(img):
    r, g, b = [img[..., k].astype(float) for k in range(3)]
    mor = (b - g) > 60                                   # mavi ≫ yeşil  →  mor çekirdek
    mor = morphology.remove_small_objects(mor, 300)
    mor = morphology.binary_closing(mor, morphology.disk(5))
    bolgeler = sorted(measure.regionprops(measure.label(mor)), key=lambda p: -p.area)
    return bolgeler[0].bbox if bolgeler else None        # (y1, x1, y2, x2)

def iou(a, b):   # kesişim / birleşim
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    k = ix * iy; return k / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - k)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a_, yol in zip(ax, dosyalar[3:6]):
    im = io.imread(yol); ad_ = os.path.basename(yol); a_.imshow(im); a_.axis("off")
    k = akyuvar_bul(im)
    w = etiket[(etiket.dosya == ad_) & (etiket.sinif == "WBC")]
    if len(w):
        r_ = w.iloc[0]; a_.add_patch(Rectangle((r_.xmin, r_.ymin), r_.xmax - r_.xmin, r_.ymax - r_.ymin, fill=False, ec="#22c55e", lw=2.5))
    if k:
        a_.add_patch(Rectangle((k[1], k[0]), k[3] - k[1], k[2] - k[0], fill=False, ec="yellow", lw=2.5, ls="--"))
        if len(w): a_.set_title(f"IoU = {iou((r_.xmin, r_.ymin, r_.xmax, r_.ymax), (k[1], k[0], k[3], k[2])):.2f}")
plt.suptitle("yeşil = uzman · sarı kesikli = bilgisayar (yalnız çekirdek!)"); plt.tight_layout(); plt.show()

💡 **IoU** (kesişim/birleşim) iki kutunun ne kadar örtüştüğünü 0–1 arasında ölçer. Bizim kural yalnız *çekirdeği* bulduğu için kutu
uzmanınkinden küçük kalıyor → IoU düşük. Daha iyisini derin öğrenme yapar (bölütleme modelleri: U-Net, Cellpose…).

## 8 · İlk makine öğrenmesi modelimiz: "Bu hücre ne?"
Uzmanın kutularından hücreleri kesip her biri için birkaç **özellik** çıkaralım (ortalama renk, boyut, en-boy oranı).
Sonra bir **Rastgele Orman** modeline bu özelliklerden hücre türünü öğretelim.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

ozellik = []
for ad_, grup in etiket.groupby("dosya"):
    im = io.imread(KOK + "JPEGImages/" + ad_)
    for _, r in grup.iterrows():
        p = im[r.ymin:r.ymax, r.xmin:r.xmax].reshape(-1, 3).astype(float)
        if len(p) == 0: continue
        w_, h_ = r.xmax - r.xmin, r.ymax - r.ymin
        ozellik.append([ad_, r.sinif, *p.mean(0), *p.std(0), w_ * h_, w_ / max(h_, 1)])
X = pd.DataFrame(ozellik, columns=["dosya", "sinif", "R", "G", "B", "R_std", "G_std", "B_std", "alan", "en_boy"])
print(X.shape); X.head()

In [ ]:
# Aynı görüntüdeki hücreler ya eğitime ya teste gitsin (sızıntı olmasın!)
egit, test = next(GroupShuffleSplit(test_size=0.25, random_state=0).split(X, groups=X.dosya))
oz = ["R", "G", "B", "R_std", "G_std", "B_std", "alan", "en_boy"]
model = RandomForestClassifier(n_estimators=200, random_state=0).fit(X.iloc[egit][oz], X.iloc[egit].sinif)
tahmin = model.predict(X.iloc[test][oz])
print(classification_report(X.iloc[test].sinif, tahmin, digits=3))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(X.iloc[test].sinif, tahmin, ax=ax[0], cmap="Blues", colorbar=False); ax[0].set_title("Karışıklık matrisi")
pd.Series(model.feature_importances_, index=oz).sort_values().plot.barh(ax=ax[1], color="#00426a"); ax[1].set_title("Model neye bakıyor?")
plt.tight_layout(); plt.show()

## 9 · Kendi görüntünü dene! (isteğe bağlı)
Elinde bir kan yayması (ya da benzer boyalı hücre) fotoğrafı varsa yükle; aynı sayım yöntemini onun üzerinde çalıştıralım.

In [ ]:
#@title ▶ 9 · Görüntü yükle ve say { display-mode: "form" }
try:
    from google.colab import files
    yuklenen = files.upload()
    for ad_, icerik in yuklenen.items():
        open(ad_, "wb").write(icerik)
        im = io.imread(ad_)[..., :3]
        n, lab = hucre_say(im)
        plt.figure(figsize=(7, 5)); plt.imshow(segmentation.mark_boundaries(im, lab, color=(1, 1, 0)))
        plt.title(f"{ad_}: {n} hücre (parametreleri kendi görüntüne göre ayarlaman gerekebilir)"); plt.axis("off"); plt.show()
except ImportError:
    print("Bu hücre Google Colab içinde çalışır.")

## 🎯 Toparlarsak
| Adım | Ne yaptık? | Gerçek hayatta |
|:--|:--|:--|
| Görüntü = sayı tablosu | Pikselleri okuduk | Her mikroskop fotoğrafı analiz edilebilir |
| Eşik + havza | Hücreleri saydık | Hemositometre sayımını hızlandırma |
| Uzmanla karşılaştırma | Hata ölçtük (MAE) | Yöntem doğrulama, etiket kalitesi |
| Renk kuralı | Akyuvarı bulduk (IoU) | Basit kurallar hızlı ama sınırlı |
| Rastgele Orman | Hücre türünü sınıflandırdık | Özellik → model → karar |

**Sonraki adım:** Derin öğrenme ile bölütleme (U-Net), çok odaklı mikroskopide net görüntü (EDOF) ve odaktan 3B şekil —
PharmAI KTÜ grubunun çalışmaları: https://scholar.google.com/citations?user=F2NkKNAAAAAJ

*Veri: BCCD Dataset (MIT lisansı) · https://github.com/Shenggan/BCCD_Dataset*